# Notebook 2 — Final LLM life-satisfaction prediction pipeline

This notebook sends the WVS respondent profiles produced by Notebook 1 to one selected model at a time.

**Final model set**
- GPT-5.6 Luna
- Claude Sonnet 5
- Gemini 3.8 Flash
- DeepSeek V4.1 Flash
- Qwen3.7 Plus
- Gemma 4 31B via DeepInfra

**Deepnote secrets**
`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`,
`DEEPSEEK_API_KEY`, `DASHSCOPE_API_KEY`, `DEEPINFRA_API_KEY`.

The notebook uses the same respondent profile, system instruction, and life-satisfaction question across providers. No temperature or top-p parameter is specified. DeepSeek thinking is explicitly disabled because its thinking mode can consume the short answer budget. Gemini is left with its provider-default thinking behavior and **no artificial output-token ceiling**, because the earlier 16-token ceiling caused empty visible responses in the pilot.

`PILOT_MODE = True` sends the first 100 profiles from Notebook 1's deterministic pilot file.  
`PILOT_MODE = False` uses the full preregistered N = 93,901 profile file.

Run **one model at a time** by changing only `MODEL_KEY`.


In [ ]:
# Run once in Deepnote if the packages are not already installed.
%pip install -q "openai>=1.100" "anthropic>=0.70" "google-genai>=1.0" pandas numpy tqdm


In [ ]:
from pathlib import Path
import hashlib
import os
import random
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

PILOT_MODE = False
PILOT_ROWS_PER_MODEL = 100

PILOT_INPUT = Path("output/full_wvs/profiles_for_prediction_PILOT.csv")
FULL_INPUT = Path("output/full_wvs/profiles_for_prediction.csv")
INPUT_FILE = PILOT_INPUT if PILOT_MODE else FULL_INPUT

OUTPUT_DIR = Path("output/full_wvs/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FULL_N = 93901
EXPECTED_PROFILE_VERSION = "wvs7_v6_profiles_v2_2026-09-22"
PROMPT_VERSION = "ls_prediction_v2_2026-09-22"

MAX_WORKERS = 10
BATCH_SIZE = 1000
RETRIES = 6
CHECKPOINT_EVERY = 100

# Exact Singapore workspace-specific OpenAI-compatible endpoint supplied by Alibaba Model Studio.
QWEN_BASE_URL = (
    "https://ws-pmemyycbldatjq2w.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
)

SYSTEM_PROMPT = (
    "You will be given a World Values Survey respondent profile. "
    "Answer the life-satisfaction question as that person would most likely answer it. "
    "Return ONLY one integer from 1 to 10. Do not explain your answer."
)

QUESTION = (
    "All things considered, how satisfied are you with your life as a whole these days? "
    "Using a scale on which 1 means you are 'completely dissatisfied' and 10 means you are "
    "'completely satisfied', where would you put your satisfaction with your life as a whole?"
)

def build_prompt(profile):
    return (
        f"You are a person with the following profile:\n\n{profile}"
        f"\n\nQuestion:\n{QUESTION}"
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
MODELS = {
    "openai_gpt56_luna": {
        "provider": "openai",
        "model": "gpt-5.6-luna",
        "api_key_env": "OPENAI_API_KEY",
    },
    "anthropic_sonnet5": {
        "provider": "anthropic",
        "model": "claude-sonnet-5",
        "api_key_env": "ANTHROPIC_API_KEY",
    },
    "google_gemini38_flash": {
        "provider": "google_gemini",
        "model": "gemini-3.8-flash",
        "api_key_env": "GEMINI_API_KEY",
    },
    "deepseek_v41_flash": {
        "provider": "deepseek",
        "model": "deepseek-flash",
        "api_key_env": "DEEPSEEK_API_KEY",
    },
    "alibaba_qwen37_plus": {
        "provider": "qwen",
        "model": "qwen3.7-plus",
        "api_key_env": "DASHSCOPE_API_KEY",
    },
    "deepinfra_gemma4_31b": {
        "provider": "deepinfra",
        "model": "google/gemma-4-31B-it-turbo",
        "api_key_env": "DEEPINFRA_API_KEY",
    },
}

# Change only this value to run another model.
MODEL_KEY = "alibaba_qwen37_plus"

if MODEL_KEY not in MODELS:
    raise ValueError(f"Unknown MODEL_KEY: {MODEL_KEY}")

cfg = MODELS[MODEL_KEY]
print("Selected:", MODEL_KEY, cfg)


In [ ]:
def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(
            f"Missing Deepnote secret/environment variable: {name}"
        )
    return value

def parse_score(text):
    if text is None:
        raise ValueError("Model returned no visible text.")

    text = str(text).strip()

    if re.fullmatch(r"10|[1-9]", text):
        return int(text)

    nums = re.findall(r"(?<!\d)(10|[1-9])(?!\d)", text)
    if len(nums) == 1:
        return int(nums[0])

    raise ValueError(
        f"Could not parse a unique 1-10 score from: {text!r}"
    )

_CLIENTS = {}

def reset_clients():
    global _CLIENTS
    _CLIENTS = {}

def get_client(cfg):
    provider = cfg["provider"]

    if provider in _CLIENTS:
        return _CLIENTS[provider]

    if provider == "openai":
        from openai import OpenAI
        client = OpenAI(
            api_key=require_env(cfg["api_key_env"])
        )

    elif provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic(
            api_key=require_env(cfg["api_key_env"])
        )

    elif provider == "google_gemini":
        from google import genai
        client = genai.Client(
            api_key=require_env(cfg["api_key_env"])
        )

    elif provider == "deepseek":
        from openai import OpenAI
        client = OpenAI(
            api_key=require_env(cfg["api_key_env"]),
            base_url="https://api.deepseek.com",
        )

    elif provider == "qwen":
        from openai import OpenAI
        client = OpenAI(
            api_key=require_env(cfg["api_key_env"]),
            base_url=QWEN_BASE_URL,
        )

    elif provider == "deepinfra":
        from openai import OpenAI
        client = OpenAI(
            api_key=require_env(cfg["api_key_env"]),
            base_url="https://api.deepinfra.com/v1/openai",
        )

    else:
        raise ValueError(provider)

    _CLIENTS[provider] = client
    return client

def call_model(profile, cfg):
    provider = cfg["provider"]
    model = cfg["model"]
    client = get_client(cfg)
    user_text = build_prompt(profile)

    if provider == "openai":
        r = client.responses.create(
            model=model,
            instructions=SYSTEM_PROMPT,
            input=user_text,
            reasoning={"effort": "none"},
            max_output_tokens=16,
        )
        raw = r.output_text

    elif provider == "anthropic":
        r = client.messages.create(
            model=model,
            max_tokens=16,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": user_text}
            ],
        )
        raw = "".join(
            b.text for b in r.content
            if getattr(b, "type", "") == "text"
        )

    elif provider == "google_gemini":
        from google.genai import types
        r = client.models.generate_content(
            model=model,
            contents=user_text,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
            ),
        )
        raw = r.text

    elif provider == "deepseek":
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ],
            max_tokens=16,
            extra_body={"thinking": {"type": "disabled"}},
        )
        raw = r.choices[0].message.content

    elif provider == "qwen":
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ],
            extra_body={"enable_thinking": False},
        )
        raw = r.choices[0].message.content

    elif provider == "deepinfra":
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ],
            max_tokens=16,
        )
        raw = r.choices[0].message.content

    else:
        raise ValueError(provider)

    score = parse_score(raw)
    return score, str(raw).strip()


In [ ]:
df = pd.read_csv(INPUT_FILE, low_memory=False)

required = [
    "WVS_ROW_ID", "Q49", "Q288",
    "user_description", "PROFILE_VERSION"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Rerun the revised Notebook 1 first."
    )

assert df["WVS_ROW_ID"].is_unique
assert df["Q49"].between(1, 10).all()
assert df["Q288"].between(1, 10).all()
assert df["user_description"].notna().all()

profile_versions = set(df["PROFILE_VERSION"].dropna().astype(str).unique())
if profile_versions != {EXPECTED_PROFILE_VERSION}:
    raise ValueError(
        f"Unexpected PROFILE_VERSION values: {profile_versions}. "
        f"Expected only {EXPECTED_PROFILE_VERSION!r}."
    )

INPUT_SHA256 = sha256_file(INPUT_FILE)
print("Input:", INPUT_FILE)
print("Input SHA256:", INPUT_SHA256)

if PILOT_MODE:
    if len(df) < PILOT_ROWS_PER_MODEL:
        raise ValueError("Pilot input has fewer rows than PILOT_ROWS_PER_MODEL.")
    run_df = df.head(PILOT_ROWS_PER_MODEL).copy()
    print(f"PILOT MODE: {len(run_df):,} calls for {MODEL_KEY}")
else:
    if len(df) != EXPECTED_FULL_N:
        raise ValueError(
            f"Full input has N={len(df):,}; expected N={EXPECTED_FULL_N:,}."
        )
    run_df = df.copy()
    print(f"FULL MODE: {len(run_df):,} calls for {MODEL_KEY}")

print("WVS_ROW_ID range:", int(run_df["WVS_ROW_ID"].min()),
      "to", int(run_df["WVS_ROW_ID"].max()))


## One-call connectivity test

Run this before the 100-person pilot for each provider. For Qwen, wait until Alibaba account verification/entitlement is complete. If you change a provider endpoint or API key during debugging, run `reset_clients()` (or restart the kernel) before testing again.


In [ ]:
# This makes exactly ONE API call.
test_score, test_raw = call_model(
    run_df.iloc[0]["user_description"],
    cfg
)
print("Parsed score:", test_score)
print("Raw response:", repr(test_raw))


In [ ]:
def predict_one(row, cfg):
    last_error = None

    for attempt in range(1, RETRIES + 1):
        try:
            score, raw = call_model(row["user_description"], cfg)
            return {
                "WVS_ROW_ID": int(row["WVS_ROW_ID"]),
                "prediction": int(score),
                "raw_response": raw,
                "status": "ok",
                "error": None,
                "attempts": attempt,
            }
        except Exception as e:
            last_error = repr(e)
            if attempt < RETRIES:
                wait = min(60, (2 ** (attempt - 1)) + random.random())
                time.sleep(wait)

    return {
        "WVS_ROW_ID": int(row["WVS_ROW_ID"]),
        "prediction": np.nan,
        "raw_response": None,
        "status": "error",
        "error": last_error,
        "attempts": RETRIES,
    }

def run_predictions(run_df, model_key, cfg):
    suffix = "PILOT" if PILOT_MODE else "FULL"
    out_file = OUTPUT_DIR / f"{model_key}_{suffix}.csv"

    records = []
    done = set()

    if out_file.exists():
        existing = pd.read_csv(out_file)
        records = existing.to_dict("records")

        if "status" in existing.columns:
            done = set(
                existing.loc[existing["status"].eq("ok"), "WVS_ROW_ID"]
                .dropna()
                .astype(int)
            )

        print(
            f"Resuming from {out_file}: "
            f"{len(done):,} successful rows already present."
        )

    todo = run_df.loc[~run_df["WVS_ROW_ID"].isin(done)].copy()
    print(f"Rows remaining: {len(todo):,}")

    completed_since_checkpoint = 0

    def checkpoint():
        nonlocal records
        out = pd.DataFrame(records)
        if out.empty:
            return

        out = (
            out.drop_duplicates("WVS_ROW_ID", keep="last")
            .sort_values("WVS_ROW_ID")
            .reset_index(drop=True)
        )

        out["model_key"] = model_key
        out["provider"] = cfg["provider"]
        out["model_id"] = cfg["model"]
        out["prompt_version"] = PROMPT_VERSION
        out["profile_version"] = EXPECTED_PROFILE_VERSION
        out["input_sha256"] = INPUT_SHA256

        out.to_csv(out_file, index=False)

    # Process bounded batches rather than creating ~94k futures at once.
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[start:start + BATCH_SIZE]

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(predict_one, row, cfg): int(row["WVS_ROW_ID"])
                for _, row in batch.iterrows()
            }

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{model_key} rows {start + 1}-{start + len(batch)}"
            ):
                records.append(fut.result())
                completed_since_checkpoint += 1

                if completed_since_checkpoint >= CHECKPOINT_EVERY:
                    checkpoint()
                    completed_since_checkpoint = 0

        checkpoint()

    checkpoint()
    result = pd.read_csv(out_file)

    # Restrict the final audit to IDs expected in this run.
    result = result.loc[
        result["WVS_ROW_ID"].isin(run_df["WVS_ROW_ID"])
    ].copy()

    n_ok = int(result["status"].eq("ok").sum())
    n_error = int(result["status"].eq("error").sum())

    print("Saved:", out_file)
    print(f"Successful: {n_ok:,}/{len(run_df):,}")
    print(f"Errors remaining: {n_error:,}")

    if n_ok:
        ok = result.loc[result["status"].eq("ok")].copy()
        ok["prediction"] = pd.to_numeric(ok["prediction"], errors="coerce")
        if not ok["prediction"].between(1, 10).all():
            raise AssertionError("At least one successful prediction is outside 1-10.")

    if n_error:
        print("\nRemaining errors:")
        display(
            result.loc[result["status"].eq("error"),
                       ["WVS_ROW_ID", "error"]].head(20)
        )
        raise RuntimeError(
            f"{n_error} rows still failed after retries. "
            "The checkpoint is saved. Rerun this cell to retry only failed rows."
        )

    if len(result) != len(run_df):
        raise AssertionError(
            f"Output has {len(result):,} run IDs; expected {len(run_df):,}."
        )

    return result.sort_values("WVS_ROW_ID").reset_index(drop=True)

results = run_predictions(run_df, MODEL_KEY, cfg)


In [ ]:
# Final technical audit for the selected model.
assert len(results) == len(run_df)
assert results["WVS_ROW_ID"].is_unique
assert results["status"].eq("ok").all()

pred = pd.to_numeric(results["prediction"], errors="raise")
assert pred.between(1, 10).all()
assert np.all(np.equal(pred, np.floor(pred)))

print("Technical audit passed.")
print("N =", len(results))
print("Prediction range =", int(pred.min()), "to", int(pred.max()))
print("Mean =", round(float(pred.mean()), 3))
print("SD =", round(float(pred.std(ddof=1)), 3))

# This is a pipeline diagnostic only. Do not inspect the preregistered
# income-heterogeneity/Anna Karenina relationship during the pilot.


## Run sequence

### Pilot
For each model:
1. set `MODEL_KEY`;
2. run the one-call connectivity test;
3. run the 100-row pilot;
4. require 100/100 successful integer predictions in 1–10;
5. inspect only technical diagnostics, not the preregistered substantive relationship.

The five already-working providers should be rerun because Notebook 1's profile text has changed. Qwen can be added after Alibaba verification clears.

### Full collection
After all desired pilots pass:
1. rerun Notebook 1 with `PILOT_MODE = False`;
2. set Notebook 2 `PILOT_MODE = False`;
3. confirm the input SHA256 and N = 93,901;
4. run one model at a time;
5. retain each output CSV, its `model_id`, `prompt_version`, `profile_version`, and `input_sha256`;
6. if a run stops, rerun the prediction cell: it resumes from successful IDs and retries only incomplete/failed rows.

Do not alter the profile text or prompt after full collection begins without documenting a protocol deviation.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7aaa7215-b731-433d-9b62-8be4a70a4410' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>